# Shared Loading & Functions

In [ ]:
import os, textwrap
from types import SimpleNamespace
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

save = True
COMPARE_PILOT = True     # True: only pilot (left, 'Pilot') vs. replication (right, 'Replication') figures, saved under COMPARISON_ROOT
USE_PILOT = False         # True: run everything on the pilot data instead of the replication (figures save under the pilot's figures/)

plt.rcParams.update({'font.family': 'Avenir', 'figure.dpi': 200, 'savefig.dpi': 200,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 13, 'axes.titlesize': 15, 'axes.labelsize': 14, 'figure.titlesize': 16,
                     'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12, 'legend.title_fontsize': 12})

PILOT_ROOT = '../pilot/episodic-choice-task'
DATA_ROOT = PILOT_ROOT if USE_PILOT else '..'
COMPARISON_ROOT = '../figures/pilot_comparison'  # with COMPARE_PILOT, figures save under here instead
assert not (USE_PILOT and COMPARE_PILOT), 'COMPARE_PILOT already puts the pilot next to the replication; turn off USE_PILOT'

MEM_ORDER = ['low', 'mid', 'high']
MEM_LABELS = ['Low', 'Mid', 'High']
# memorability is always light -> dark (low -> high); pick the hue per measure
MEM_PALETTES = {
    'blue':   dict(zip(MEM_ORDER, ['#9ecae1', '#3182bd', '#08519c'])),
    'purple': dict(zip(MEM_ORDER, ['#bcbddc', '#756bb1', '#54278f'])),
    'green':  dict(zip(MEM_ORDER, ['#a1d99b', '#31a354', '#006d2c'])),
}
VALUE_PALETTE = {0.0: '#fdae6b', 1.0: '#cc4c02'}  # $0 light orange, $1 burnt orange
GRAY_PALETTE = {'low': '#bdbdbd', 'mid': '#969696', 'high': '#525252'}  # incorrect trials, light -> dark
# RT by memorability: incorrect in gray, correct in purple, both light -> dark (low -> high memorability); a spec adds split_by
correct_split = dict(split_order=[0.0, 1.0], split_labels=['Incorrect', 'Correct'],
                     split_palettes=[GRAY_PALETTE, MEM_PALETTES['purple']],
                     split_line_colors=[GRAY_PALETTE['mid'], MEM_PALETTES['purple']['mid']])


def marker_style(n_lines):
    """Black-ringed markers; fewer mean lines on the plot -> bigger markers."""
    return dict(markeredgecolor='black', markeredgewidth=1, markersize={1: 12, 2: 10, 3: 8}[n_lines])


MAIN_FIGSIZE = (3.35, 4.55)  # every single-panel figure (smallest size exp1's legends clear the data at)
SHORT_AXES_SIZE = (2.65, 2.8)  # axes (not figure) size of the P(choose right) figures, so they match
COMPARE_FIGSIZE = (6.06, 4.81)  # every pilot | replication figure (exp1's); 4-panel ones are twice as wide

# every figure spec starts from these; a spec only lists what it changes. Figures always show the included sample.
# ylim=(bottom, top): None = automatic.
FIG_DEFAULTS = dict(
    figsize=MAIN_FIGSIZE,    # size of one panel
    ylim=(None, None),
    ytick_step=0.1,          # y ticks on multiples of this (probabilities, RT in s)
    chance_color='gray',
    errorbar=('ci', 95),     # error bars: 95% CI of the per-subject means ('se' for standard error)
    **marker_style(1),       # override with **marker_style(2) / (3) when a plot has 2 / 3 lines
)

"""COLUMNS
'experiment_id','participant_id', 'is_choice_trial','is_attention_check','rt','response', 'correct','trial_number','block_index','old_trial',
'memorability_bin','encoding_trial','delay','old_side','old_value','left_image_name','left_image_path','left_memorability','left_value',
'left_is_old','right_image_name','right_image_path','right_memorability','right_value','right_is_old','chosen_side','chosen_image_name',
'chosen_image_path','chosen_value','reward','repeat_source_was_chosen','response_key','choice_missed','old_chosen','optimal_old_choice','final_bonus'
"""

In [ ]:
### LOADING & EXCLUSIONS

def load_data(label, data_root=DATA_ROOT):
    if label.startswith('food_choice/'):  # 'food_choice/gap23' -> food_choice_data_gap23.csv, figures in food_choice/gap23/
        fname = f"food_choice_data_{label.split('/')[1]}.csv"
    else:
        fname = {'main': 'episodic_choice_data.csv'}.get(label, f'episodic_choice_data-{label}.csv')
    df = pd.read_csv(f'{data_root}/data/{fname}')
    excluded_path = f'{data_root}/data/excluded/{fname}'  # excluded participants are kept in their own file
    if os.path.exists(excluded_path):
        df = pd.concat([df, pd.read_csv(excluded_path)], ignore_index=True)
    figures_dir = f'{COMPARISON_ROOT}/{label}' if COMPARE_PILOT else f'{data_root}/figures/{label}'
    os.makedirs(figures_dir, exist_ok=True)
    return df, figures_dir


def _fit_axes_size(fig, size, tight):
    """Resize fig until every axes is size = (width, height) inches; margins, labels and titles keep their size."""
    for _ in range(5):
        if tight:
            plt.tight_layout()
        fig.canvas.draw()
        boxes = [a.get_window_extent() for a in fig.get_axes() if a.get_visible()]
        n_cols, n_rows = len({round(b.x0) for b in boxes}), len({round(b.y0) for b in boxes})
        w, h = boxes[0].size / fig.dpi
        if abs(w - size[0]) < 0.004 and abs(h - size[1]) < 0.004:
            return
        fig.set_size_inches(fig.get_figwidth() + n_cols * (size[0] - w), fig.get_figheight() + n_rows * (size[1] - h))


def finish_fig(figures_dir, fname, tight=True, axes_size=None):
    """Save (if save) and display. axes_size = (width, height) in inches resizes the figure to give its axes that size."""
    if axes_size is not None:
        _fit_axes_size(plt.gcf(), axes_size, tight)
    elif tight:
        plt.tight_layout()
    if save:
        plt.savefig(os.path.join(figures_dir, fname))
    plt.show()


def find_ai_pids(df):
    """Participants who pressed the hidden key on attention checks (white-text AI check).
    Old (episodic_choice_v3) data used the space bar; everything else uses 'x'."""
    attention_df = df.query('is_attention_check == True')
    is_old_format = attention_df['experiment_id'] == 'episodic_choice_v3'
    hidden_df = pd.concat([
        attention_df[is_old_format].query('response_key == " "'),
        attention_df[~is_old_format].query('response_key == "x" or response == "x"'),
    ])
    print(f'{len(hidden_df)} hidden-key (space or x) responses on attention checks')
    print(hidden_df[['participant_id', 'trial_number', 'response', 'response_key', 'correct']])
    ai_pids = hidden_df['participant_id'].unique().tolist()

    if ai_pids:
        choice_df = df[df['participant_id'].isin(ai_pids) & (df['is_choice_trial'] == True)]
        if 'optimal_old_choice' in choice_df:  # not in the food-choice task
            print(f"Suspected AI optimal old choice rate: {pd.to_numeric(choice_df['optimal_old_choice'], errors='coerce').mean():.3f}")
    return ai_pids


def find_failed_attention_pids(df, threshold=1):
    """Participants below threshold accuracy on the attention checks (default: failed any)."""
    accuracy = df.query('is_attention_check == True').groupby('participant_id')['correct'].mean()
    return accuracy[accuracy < threshold].index.tolist()


def find_incomplete_pids(df, trial_query, min_trials):
    counts = df.query(trial_query).groupby('participant_id').size()
    return counts[counts < min_trials].index.tolist()


def find_high_miss_pids(df, threshold=0.2):
    """Preregistered: no response on > threshold of all response trials (value-report prompts count in the direct task)."""
    response_df = df.query('is_choice_trial == True')
    no_col = pd.Series(False, index=response_df.index)
    is_value_test = response_df.get('is_value_test_trial', no_col) == True
    is_missed = lambda s: s.map(lambda v: str(v).lower() in ('true', '1', '1.0'))
    missed = ((is_missed(response_df.get('value_test_missed', no_col)) & is_value_test) |
              (is_missed(response_df['choice_missed']) & ~is_value_test))
    miss_rate = missed.groupby(response_df['participant_id']).mean()
    return miss_rate[miss_rate > threshold].index.tolist()


def binom_test_by_subject(data, col):
    """One-sample binomial test per subject against chance (0.5), one-tailed."""
    rows = []
    for pid, vals in data.dropna(subset=[col]).groupby('participant_id')[col]:
        p = stats.binomtest(int(vals.sum()), len(vals), 0.5, alternative='greater').pvalue
        rows.append({'participant_id': pid, col: vals.mean(), 'pval': p, 'significant': p < 0.05})
    return pd.DataFrame(rows)


def plot_overall_performance(df, perf_query, perf_col, perf_label, ai_pids, failed_pids):
    """Per-subject performance bars (non-significant first), shown but never saved; returns the
    significantly-above-chance pids."""
    data = df.query(perf_query).copy()
    data[perf_col] = pd.to_numeric(data[perf_col], errors='coerce')
    plot_df = binom_test_by_subject(data, perf_col)
    plot_df = plot_df.sort_values(['significant', perf_col]).reset_index(drop=True)
    plot_df['x'] = np.arange(len(plot_df))
    print(f"{int(plot_df['significant'].sum())} out of {len(plot_df)} participants significantly above chance "
          f"on {perf_col} (one-sample binomial test, p < 0.05, one-tailed)")
    sig_pids = plot_df.query('significant')['participant_id'].tolist()
    if COMPARE_PILOT:  # comparison runs only make pilot-vs-replication figures
        return sig_pids

    def bar_color(row):
        if row['participant_id'] in ai_pids:
            return 'mediumpurple'
        elif row['participant_id'] in failed_pids:
            return 'tomato'
        elif row['significant']:
            return 'steelblue'
        return 'gray'

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(plot_df['x'], plot_df[perf_col], color=[bar_color(row) for _, row in plot_df.iterrows()])
    ax.set_title(f"Overall {perf_label}\n(n={plot_df['participant_id'].nunique()})")
    ax.set_xlabel('Participant')
    ax.set_ylabel(perf_label)
    ax.set_xticks(plot_df['x'])
    ax.set_xticklabels(plot_df['participant_id'], rotation=30, ha='right', fontsize=3)
    ax.axhline(y=0.5, linestyle='dashed', color='red')
    ax.set_ylim(0, 1)

    # vertical line between non-significant and significant participants
    n_not_sig = int((~plot_df['significant']).sum())
    if 0 < n_not_sig < len(plot_df):
        ax.axvline(x=n_not_sig - 0.5, color='black', linestyle='--', linewidth=1.5)

    ax.legend(handles=[
        Patch(facecolor='steelblue', label='Significant above chance'),
        Patch(facecolor='gray', label='Not significant'),
        Patch(facecolor='tomato', label='Failed attention check'),
        Patch(facecolor='mediumpurple', label='Suspected AI'),
    ], loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()
    return sig_pids


def exclude(exp, pids):
    """Add pids to the experiment's exclusions and recompute the included sample."""
    exp.excluded_pids = exp.excluded_pids | set(pids)
    exp.included_pids = [pid for pid in exp.sig_pids if pid not in exp.excluded_pids]


def setup_experiment(label, perf_query='old_trial == 1', perf_col='optimal_old_choice',
                     perf_label='Old Trial Optimal Performance', min_trials=None, data_root=DATA_ROOT):
    """Load one task's data and run the shared exclusion checks.

    Included participants are significantly above chance on perf_col (one-tailed binomial test),
    are not suspected AI, pass every attention check, respond on at least 80% of response trials, and
    (if min_trials is given) have at least min_trials trials matching perf_query. Same criteria as
    scripts/count_participants.py."""
    source = 'Pilot' if data_root == PILOT_ROOT else 'Replication'
    print(f'===== {label} ({source}) =====')
    df, figures_dir = load_data(label, data_root)
    ai_pids = find_ai_pids(df)
    failed_pids = find_failed_attention_pids(df)
    incomplete_pids = find_incomplete_pids(df, perf_query, min_trials) if min_trials else []
    high_miss_pids = find_high_miss_pids(df)
    print(f'{len(high_miss_pids)} participants with > 20% missed responses')
    sig_pids = plot_overall_performance(df, perf_query, perf_col, perf_label, ai_pids, failed_pids)

    exp = SimpleNamespace(label=label, source=source, df=df, figures_dir=figures_dir, ai_pids=ai_pids, failed_pids=failed_pids,
                          incomplete_pids=incomplete_pids, high_miss_pids=high_miss_pids, sig_pids=sig_pids,
                          excluded_pids=set())
    exclude(exp, ai_pids + failed_pids + incomplete_pids + high_miss_pids)
    print(f'Included n = {len(exp.included_pids)}')
    return exp

In [ ]:
### FIGURES 

def _draw_panels(panels, plot_fn, title, panel_size, figures_dir, fname, sharey, **kwargs):
    """One row of panels, panels = [(data, panel_title)]; with pilot data (USE_PILOT / COMPARE_PILOT) every title
    gets its n on a second line.
    A lone untitled panel takes the figure title itself; otherwise it goes above the row. Legend on the last panel
    (or legend_panel); compare_figsize overrides COMPARE_FIGSIZE for a pilot comparison."""
    w, h = panel_size
    fig, axes = plt.subplots(1, len(panels), figsize=(w * len(panels), h), sharey=sharey, squeeze=False)
    axes = axes[0]
    ylim = kwargs.pop('ylim', None)  # set once every panel is drawn: with a shared y, setting it early freezes it
    legend_opts = kwargs.pop('legend_opts', {})  # e.g. dict(loc='lower right', fontsize=10) pins the legend
    axes_size = kwargs.pop('axes_size', None)  # (width, height) each axes is resized to
    legend_ax = axes[kwargs.pop('legend_panel', -1)]  # the panel that keeps the legend (default: the last)
    compare_figsize = kwargs.pop('compare_figsize', COMPARE_FIGSIZE)  # pilot-comparison size (two panels)
    n_line = lambda n: f'\n(n={n})' if USE_PILOT or COMPARE_PILOT else ''
    for i, (ax, (panel_data, panel_title)) in enumerate(zip(axes, panels)):
        n = plot_fn(ax, panel_data, **kwargs)
        if panel_title is not None:
            ax.set_title(panel_title + ('' if 'n=' in panel_title else n_line(n)))  # a title can give its own n
        if sharey and i > 0:
            ax.set_ylabel('')
        if ax.get_legend() is not None and ax is not legend_ax:
            ax.get_legend().remove()
    if ylim is not None and ylim != (None, None):
        for ax in axes:
            ax.set_ylim(*ylim)
    # ~7 title chars per inch; a title with its own line breaks keeps them
    solo_title = (title if '\n' in title else textwrap.fill(title, int(w * 7))) + n_line(n)
    if len(panels) == 1 and panels[0][1] is None:
        axes[0].set_title(solo_title)
    else:
        fig.suptitle(title)
    # resize so each axes matches a solo panel's (w x h, legend aside): inner shared-y panels drop the y label/ticks
    # (width), the suptitle + panel titles take more room than the solo title (height), and the legend adds width
    plt.tight_layout()
    renderer = fig.canvas.get_renderer()
    first, last = axes[0].get_window_extent(renderer), axes[-1].get_window_extent(renderer)
    ignored = [a for a in (legend_ax.get_legend(), axes[0].title, axes[-1].title) if a is not None]  # not fixed margin
    for artist in ignored:
        artist.set_visible(False)
    overhang = (first.x0 - axes[0].get_tightbbox(renderer).x0 + axes[-1].get_tightbbox(renderer).x1 - last.x1) / fig.dpi
    for artist in ignored:
        artist.set_visible(True)
    pad = 1.08 * plt.rcParams['font.size'] / 72  # tight_layout's default edge pad, in inches
    solo_width = w - overhang - 2 * pad
    top = fig.get_figheight() - first.y1 / fig.dpi - pad
    panel_title = axes[0].get_title()
    axes[0].set_title(solo_title)
    solo_top = (axes[0].get_tightbbox(renderer).y1 - first.y1) / fig.dpi
    axes[0].set_title(panel_title)
    fig.set_size_inches(fig.get_figwidth() + len(axes) * (solo_width - first.width / fig.dpi),
                        fig.get_figheight() + top - solo_top)
    if COMPARE_PILOT and sharey and len(axes) > 1:  # pilot comparisons: one fixed size, whatever the titles / ticks
        fig.set_size_inches(compare_figsize[0] * len(axes) / 2, compare_figsize[1])
    if legend_ax.get_legend() is not None:  # inside its panel
        legend_opts = dict(legend_opts)
        _place_legend_inside(fig, legend_ax, [legend_opts.pop('loc')] if 'loc' in legend_opts else LEGEND_LOCS,
                             **legend_opts)
    finish_fig(figures_dir, fname, axes_size=axes_size)


LEGEND_LOCS = ['upper left', 'upper right', 'lower right', 'lower left',
               'center left', 'center right', 'upper center', 'lower center']


def _place_legend_inside(fig, ax, locs=LEGEND_LOCS, grow=1.1, max_tries=10, allow_overlap=False, **legend_kws):
    """Put ax's legend at the first of locs clear of every line, marker and error bar. If none is clear, enlarge
    the figure (text stays the same size, so the data spreads out under the legend) and try again; with
    allow_overlap, keep the size and let matplotlib pick the least-overlapping spot (or keep a single given loc)."""
    renderer = fig.canvas.get_renderer()
    for _ in range(max_tries):
        plt.tight_layout()
        points = _data_points(ax)
        for loc in locs:
            sns.move_legend(ax, loc, **legend_kws)
            box = ax.get_legend().get_window_extent(renderer).padded(6 * fig.dpi / 72)  # ~marker radius of room
            if not box.count_contains(points):
                return
        if allow_overlap:
            sns.move_legend(ax, locs[0] if len(locs) == 1 else 'best', **legend_kws)
            return
        fig.set_size_inches(fig.get_size_inches() * grow)
    print(f'legend still overlaps the data after growing the figure {grow}x{max_tries}: {ax.get_title()!r}')


def _data_points(ax, per_segment=50):
    """Display coords densely along every line (error bars included) and at every scatter point of ax."""
    points = []
    for line in ax.lines:
        xy = line.get_transform().transform(line.get_xydata())
        xy = xy[np.isfinite(xy).all(axis=1)]
        points.append(xy)
        for a, b in zip(xy[:-1], xy[1:]):
            points.append(a + np.linspace(0, 1, per_segment)[:, None] * (b - a))
    for coll in ax.collections:
        points.append(coll.get_offset_transform().transform(coll.get_offsets()))
    return np.concatenate([p for p in points if len(p)]) if points else np.empty((0, 2))


def _draw_panel_groups(groups, group_titles, col_titles, plot_fn, title, figsize, figures_dir, fname, **kwargs):
    """groups[g][c] = panel data: one row of panels in side-by-side groups, all sharing the y axis. Each group has one
    title across its panels (group_titles), above the panels' own titles (col_titles, with each panel's n). y label
    and tick labels on the first panel only, legend on the last. Groups are sized so every panel's axes are the same
    width, though only the first group carries the y label."""
    ngroups, ncols = len(groups), len(groups[0])
    ylim = kwargs.pop('ylim', None)
    kwargs.pop('legend_opts', None)
    axes_size = kwargs.pop('axes_size', None)

    def draw(width_ratios):
        fig = plt.figure(figsize=figsize, layout='constrained')
        fig.suptitle(title)
        subfigs = fig.subfigures(1, ngroups, width_ratios=width_ratios)
        axes = np.array([subfig.subplots(1, ncols, squeeze=False)[0] for subfig in subfigs])
        for subfig, group_title in zip(subfigs, group_titles):
            subfig.suptitle(group_title)
        for g in range(ngroups):
            for c in range(ncols):
                ax = axes[g, c]
                if (g, c) != (0, 0):
                    ax.sharey(axes[0, 0])
                n = plot_fn(ax, groups[g][c], **kwargs)
                ax.set_title(f'{col_titles[c]}\n(n={n})')
                if (g, c) != (0, 0):
                    ax.set_ylabel('')
                    ax.tick_params(labelleft=False)
                if (g, c) != (ngroups - 1, ncols - 1) and ax.get_legend() is not None:
                    ax.get_legend().remove()
        if ylim is not None and ylim != (None, None):
            axes[0, 0].set_ylim(*ylim)
        return fig, subfigs, axes

    # draw once with equal groups, then again with each group's label overhead added to a common axes width
    fig, subfigs, axes = draw([1] * ngroups)
    fig.canvas.draw()
    axes_w = np.array([sum(ax.get_window_extent().width for ax in axes[g]) for g in range(ngroups)])
    group_w = np.array([subfig.bbox.width for subfig in subfigs])
    plt.close(fig)
    draw(list(group_w - axes_w + axes_w.mean()))
    finish_fig(figures_dir, fname, tight=False, axes_size=axes_size)  # constrained layout already placed everything


def plot_included(sources, figures_dir, plot_fn, title, fname, figsize=MAIN_FIGSIZE,
                  panel_by=None, panel_order=None, panel_titles=None, panel_groups=False, **kwargs):
    """Included participants only, saved as fname: one panel (figsize each) per source (exp, data, label), or with
    panel_by one panel per panel_order level for each source. All panels share the y axis. With panel_groups and more
    than one source, the panels are grouped by panel_order level instead (each source side by side within a level),
    each group titled by panel_groups (a list of group titles) or else panel_titles."""
    if panel_groups and panel_by is not None and len(sources) > 1:
        groups = [[data[data['participant_id'].isin(exp.included_pids) & (data[panel_by] == level)]
                   for exp, data, _ in sources] for level in panel_order]
        group_titles = panel_groups if isinstance(panel_groups, (list, tuple)) else panel_titles
        return _draw_panel_groups(groups, group_titles, [label for _, _, label in sources], plot_fn, title,
                                  (COMPARE_FIGSIZE[0] * len(panel_order), COMPARE_FIGSIZE[1]),
                                  figures_dir, fname, **kwargs)
    panels = []
    for exp, data, label in sources:
        data = data[data['participant_id'].isin(exp.included_pids)]
        if panel_by is None:
            panels.append((data, label))
        else:
            panels += [(data[data[panel_by] == level], f'{label}\n{panel_title}' if label else panel_title)
                       for level, panel_title in zip(panel_order, panel_titles)]
    _draw_panels(panels, plot_fn, title, figsize, figures_dir, fname, sharey=True, **kwargs)


def plot_figures(build_figures, exp, pilot=None):
    """build_figures(exp) -> figure specs, each laid over FIG_DEFAULTS and drawn for the included participants.
    With COMPARE_PILOT, each spec is also built from the pilot's data and drawn to the left ('Pilot' | 'Replication');
    nothing is drawn without a pilot to compare. A spec's compare_title, if any, replaces its title there."""
    if COMPARE_PILOT and pilot is None:
        return
    figures = build_figures(exp)
    pilot_figures = build_figures(pilot) if COMPARE_PILOT else [None] * len(figures)
    for fig, pilot_fig in zip(figures, pilot_figures):
        fig = {**FIG_DEFAULTS, **fig}
        data = fig.pop('data')
        compare_title = fig.pop('compare_title', None)
        axes_like = fig.pop('axes_like', None)  # fname of a spec whose axes size this one's axes are resized to
        if axes_like:
            fig['axes_size'] = _standalone_axes_size(build_figures, exp, pilot, axes_like)
        fig.pop('panel_title', None)  # only plot_side_by_side uses it
        sources = [(exp, data, None)]
        if COMPARE_PILOT:
            sources = [(pilot, pilot_fig['data'], 'Pilot'), (exp, data, 'Replication')]
            fig['title'] = compare_title or fig['title']
        plot_included(sources, exp.figures_dir, **fig)


def _standalone_axes_size(build_figures, exp, pilot, fname):
    """(width, height) in inches of one axes in the figure plot_figures draws for the spec with this fname (not saved)."""
    global finish_fig
    real_finish_fig, sizes = finish_fig, []

    def measure(figures_dir, fname, tight=True, axes_size=None):
        if tight:
            plt.tight_layout()
        fig = plt.gcf()
        fig.canvas.draw()
        sizes.append(fig.get_axes()[0].get_window_extent().size / fig.dpi)
        plt.close(fig)

    finish_fig = measure
    try:
        plot_figures(lambda e: [f for f in build_figures(e) if f['fname'] == fname], exp, pilot)
    finally:
        finish_fig = real_finish_fig
    return sizes[0]


def plot_side_by_side(build_figures, exp, pilot, fnames, title, fname, share_ylabel=True, legends='last'):
    """Some of build_figures' specs (picked by fname) as side-by-side panels, each with its own y axis, under one title
    and with the y label once, on the left. With COMPARE_PILOT each spec becomes a group of pilot | replication panels
    (spec title above the group); otherwise the spec title is the panel title. A spec's panel_title, if any, replaces
    its title here. Every axes is sized to match the one in that spec's own figure. share_ylabel: y label on the
    first panel only (else every spec keeps its own); legends: 'last' for the last panel's legend only, 'all' for
    each spec's own."""
    if COMPARE_PILOT and pilot is None:
        return
    with_defaults = lambda e: {f['fname']: {**FIG_DEFAULTS, **f} for f in build_figures(e)}
    specs = [with_defaults(exp)[f] for f in fnames]
    pilot_specs = [with_defaults(pilot)[f] for f in fnames] if COMPARE_PILOT else None
    targets = [_standalone_axes_size(build_figures, exp, pilot, f) for f in fnames]

    def draw(figsize, width_ratios):
        fig = plt.figure(figsize=figsize, layout='constrained')
        fig.suptitle(title)
        subfigs = fig.subfigures(1, len(specs), width_ratios=width_ratios)
        groups = []
        for k, (subfig, spec) in enumerate(zip(subfigs, specs)):
            sources = ([(pilot, pilot_specs[k], 'Pilot'), (exp, spec, 'Replication')] if COMPARE_PILOT
                       else [(exp, spec, None)])
            axes = np.atleast_1d(subfig.subplots(1, len(sources), sharey=True))
            for ax, (src_exp, src_spec, label) in zip(axes, sources):
                kwargs = {key: v for key, v in src_spec.items()
                          if key not in ('data', 'plot_fn', 'title', 'panel_title', 'compare_title', 'fname', 'figsize',
                                         'legend_opts', 'ylim')}
                data = src_spec['data'][src_spec['data']['participant_id'].isin(src_exp.included_pids)]
                n = src_spec['plot_fn'](ax, data, **kwargs)
                if label is None:
                    ax.set_title(spec.get('panel_title', spec['title']) + (f'\n(n={n})' if USE_PILOT else ''))
                else:
                    ax.set_title(f'{label}\n(n={n})')
                if ax is not axes[0] or (k > 0 and share_ylabel):
                    ax.set_ylabel('')
                if ax.get_legend() is not None and not (ax is axes[-1] and (legends == 'all' or k == len(specs) - 1)):
                    ax.get_legend().remove()
            if spec['ylim'] != (None, None):  # once every panel is drawn: with a shared y, setting it early freezes it
                axes[0].set_ylim(*spec['ylim'])
            if COMPARE_PILOT:
                subfig.suptitle(spec.get('panel_title', spec['title']))
            groups.append(axes)
        for spec, axes in zip(specs, groups):
            if axes[-1].get_legend() is not None:  # placed as the spec asks
                legend_opts = spec.get('legend_opts', {})
                sns.move_legend(axes[-1], legend_opts.get('loc', 'best'),
                                **{k: v for k, v in legend_opts.items() if k in ('fontsize', 'title_fontsize')})
        return fig, subfigs, groups

    # draw, measure, and redraw: each pass grows every group by how far its axes are from the standalone width and the
    # figure by how far off the axes height is (constrained layout's label / tick overhead shifts a little each time)
    figsize = (2 * COMPARE_FIGSIZE[0], COMPARE_FIGSIZE[1]) if COMPARE_PILOT else (2 * MAIN_FIGSIZE[0], MAIN_FIGSIZE[1])
    widths = [figsize[0] / len(specs)] * len(specs)
    for attempt in range(4):
        fig, subfigs, groups = draw((sum(widths), figsize[1]), widths)
        if attempt == 3:
            break
        fig.canvas.draw()
        sizes = [[ax.get_window_extent().size / fig.dpi for ax in axes] for axes in groups]
        widths = [width + sum(w - size[0] for size in group_sizes)
                  for width, group_sizes, (w, _) in zip(widths, sizes, targets)]
        figsize = (sum(widths), figsize[1] + targets[0][1] - sizes[0][0][1])
        plt.close(fig)
    finish_fig(exp.figures_dir, fname, tight=False)  # constrained layout already placed everything


# axis options any plot spec can set; plot functions route these to _style_ax instead of seaborn
STYLE_KEYS = ('xlabel', 'ylabel', 'xlim', 'ylim', 'xticks', 'xticklabels', 'yticks', 'yticklabels', 'ytick_step')


def _pop_style(kwargs):
    return {k: kwargs.pop(k) for k in STYLE_KEYS if k in kwargs}


def _style_ax(ax, chance=None, chance_color='gray', xlabel=None, ylabel=None, xlim=None, ylim=None,
              xticks=None, xticklabels=None, yticks=None, yticklabels=None, ytick_step=0.1):
    if chance is not None:
        ax.axhline(chance, linestyle='dashed', color=chance_color)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if xticks is not None:
        ax.set_xticks(xticks, labels=xticklabels)
    if yticks is not None:
        ax.set_yticks(yticks, labels=yticklabels)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None and ylim != (None, None):  # set_ylim freezes autoscaling, even for a None end
        ax.set_ylim(*ylim)
    if yticks is None and np.ptp(ax.get_ylim()) <= 2:  # probabilities and RTs in s: ticks on multiples of ytick_step
        ax.yaxis.set_major_locator(MultipleLocator(ytick_step))


def _markers_on_top(ax):
    """Draw the point markers (and their edges) over the error bars."""
    for line in ax.lines:
        if line.get_marker() not in (None, '', 'None'):
            line.set_zorder(3)


def plot_subject_lines(ax, data, x, y, order=None, color='steelblue', errorbar=('ci', 95), chance=0.5, chance_color='gray', **kwargs):
    """Per-subject means as light gray lines, with the group mean +/- SE on top."""
    style = _pop_style(kwargs)
    plot_df = data.groupby(['participant_id', x], as_index=False).agg({y: 'mean'})
    if order is not None:
        # otherwise the lineplot fixes the x order by first appearance, ignoring `order`
        plot_df[x] = pd.Categorical(plot_df[x], categories=order, ordered=True)
    sns.lineplot(x=x, y=y, units='participant_id', estimator=None, data=plot_df, color='lightgray', ax=ax)
    sns.pointplot(x=x, y=y, data=plot_df, errorbar=errorbar, color=color, order=order, ax=ax, **kwargs)
    _markers_on_top(ax)
    _style_ax(ax, chance, chance_color, **style)
    return plot_df['participant_id'].nunique()


def plot_hue_means(ax, data, x, y, hue, order=None, hue_order=None, palette=None, hue_labels=None,
                   legend_title=None, dodge=0.3, errorbar=('ci', 95), extra_by=(), legend_reverse=False,
                   chance=None, chance_color='gray', **kwargs):
    """Group mean +/- error bar of per-subject means, split by hue. extra_by: extra columns to
    average within before taking each subject's mean. legend_reverse: list the hue levels in reverse order (e.g. to
    keep the legend low -> high when a reversed hue_order flips the dodge)."""
    style = _pop_style(kwargs)
    plot_df = data.groupby(['participant_id', x, hue, *extra_by], as_index=False).agg({y: 'mean'})
    sns.pointplot(x=x, y=y, hue=hue, data=plot_df, errorbar=errorbar, order=order, hue_order=hue_order,
                  palette=palette, dodge=dodge, ax=ax, **kwargs)
    _markers_on_top(ax)
    if hue_labels is not None or legend_title is not None:
        handles, labels = ax.get_legend_handles_labels()
        labels = hue_labels or labels
        if legend_reverse:
            handles, labels = handles[::-1], labels[::-1]
        ax.legend(handles=handles, labels=labels, title=legend_title)
    _style_ax(ax, chance, chance_color, **style)
    return plot_df['participant_id'].nunique()


def plot_colored_means(ax, data, x, y, order, palette, errorbar=('ci', 95), line_color='black',
                       split_by=None, split_order=None, split_palettes=None, split_line_colors=None,
                       split_labels=None, legend_title=None, dodge=0.08, split_markers=None, split_linestyles=None,
                       split_open=None, line_width=None, legend_colors=None, chance=None, chance_color='gray', **kwargs):
    """Group mean +/- error bar of per-subject means, one color per x level, joined by a plain line.
    split_by: one such line per split_order level, each with its own palette / line color (and optionally marker
    shape / line style, or open markers: white-filled), dodged apart. line_width: of the joining lines (default 2);
    legend_colors: fill of each split's legend marker (default its line color; open markers stay white)."""
    style = _pop_style(kwargs)
    plot_df = data.groupby(['participant_id', x, *([split_by] if split_by else [])], as_index=False).agg({y: 'mean'})
    if split_by is None:
        lines = [(plot_df, palette, line_color, 0.0, 'o', '-', False)]
    else:
        offsets = np.linspace(-dodge, dodge, len(split_order))
        markers = split_markers or ['o'] * len(split_order)
        linestyles = split_linestyles or ['-'] * len(split_order)
        opens = split_open or [False] * len(split_order)
        lines = [(plot_df[plot_df[split_by] == level], pal, lc, off, marker, ls, is_open)
                 for level, pal, lc, off, marker, ls, is_open
                 in zip(split_order, split_palettes, split_line_colors, offsets, markers, linestyles, opens)]
    for line_df, pal, lc, off, marker, ls, is_open in lines:
        n_before = len(ax.lines)
        sns.pointplot(x=x, y=y, hue=x, data=line_df, order=order, hue_order=order, palette=pal,
                      errorbar=errorbar, legend=False, marker=marker, ax=ax, **kwargs)
        for ln in ax.lines[n_before:]:
            ln.set_xdata(np.asarray(ln.get_xdata(), dtype=float) + off)
            if is_open and ln.get_marker() not in (None, '', 'None'):
                ln.set_markerfacecolor('white')
        means = line_df.groupby(x)[y].mean().reindex(order)
        ax.plot(np.arange(len(order)) + off, means.values, color=lc, linestyle=ls, linewidth=line_width or 2, zorder=1)
    if split_by is not None:
        marker_kws = {k: v for k, v in kwargs.items() if k.startswith('marker')}
        handles = [Line2D([], [], color=lc, marker=marker, linestyle=ls, **({'linewidth': line_width} if line_width else {}),
                          **{**marker_kws, 'markerfacecolor': 'white' if is_open else face})
                   for lc, marker, ls, is_open, face
                   in zip(split_line_colors, markers, linestyles, opens, legend_colors or split_line_colors)]
        ax.legend(handles=handles, labels=split_labels, title=legend_title)
    _markers_on_top(ax)
    _style_ax(ax, chance, chance_color, **style)
    return plot_df['participant_id'].nunique()


# x ticks shared across experiments
no_yes = dict(xticks=[0, 1], xticklabels=['No', 'Yes'])
correct_ticks = dict(xticks=[0, 1], xticklabels=['Incorrect', 'Correct'])
side_ticks = dict(xticks=[0, 1], xticklabels=['Left>Right', 'Right>Left'], xlim=(-0.5, 1.5))  # which card is higher

In [ ]:
### UTILITIES (unused)

def print_bonuses(df):
    """participant_id,final_bonus for bonus payment."""
    bonus_df = df.groupby('participant_id', as_index=False).agg({'final_bonus': 'max'}).sort_values('participant_id')
    print('For bonus payment: ')
    for _, row in bonus_df.iterrows():
        print(f'{row.participant_id},{row.final_bonus}')


def plot_rt_distributions(df, figures_dir, col_wrap=10):
    g = sns.displot(x='rt', data=df.query('is_choice_trial == True'), col='participant_id', col_wrap=col_wrap)
    g.figure.suptitle(f"All Choice Trials\n(n={df['participant_id'].nunique()})", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

    g = sns.displot(x='rt', data=df.query('old_trial == 1'), col='participant_id', col_wrap=col_wrap)
    g.figure.suptitle(f"Old Trials\n(n={df['participant_id'].nunique()})", fontsize=16, y=1.02)
    finish_fig(figures_dir, 'rt_dist.png')

# Experiment 1 (Main)

In [ ]:
exp1 = setup_experiment('main')
exp1_pilot = setup_experiment('main', data_root=PILOT_ROOT) if COMPARE_PILOT else None
# print_bonuses(exp1.df)

In [ ]:
value_ticks = dict(xticks=[0, 1], xticklabels=['$0', '$1'], xlim=(-0.2, 1.2))


def exp1_old_trials(exp):
    old_trials = exp.df.query('old_trial == 1').dropna(subset=['optimal_old_choice']).copy()
    old_trials['rt_sec'] = old_trials['rt'] / 1000
    return old_trials


def exp1_figures(exp):
    old_trials = exp1_old_trials(exp)
    mem_choices = dict(data=old_trials, plot_fn=plot_hue_means, title='Experiment 1A', ylim=(0.3, 0.8),
                       x='old_value', y='old_chosen', hue='memorability_bin', hue_order=MEM_ORDER, dodge=False,
                       palette=MEM_PALETTES['blue'], **marker_style(3), native_scale=True, chance=0.5, **value_ticks,
                       hue_labels=MEM_LABELS, legend_title='Memorability Bin',
                       legend_opts=dict(loc='lower right', fontsize=10, title_fontsize=10, grow=1.02),
                       xlabel='Old Card Value', ylabel='P(Choose Old Card)')
    return [
        dict(data=old_trials, plot_fn=plot_hue_means, title='Old Trial Choices by Delay', fname='old_choices_delay.png',
             figsize=(5, 4), x='delay', y='old_chosen', hue='old_value', hue_order=[0.0, 1.0], palette=VALUE_PALETTE,
             hue_labels=['$0', '$1'], legend_title='Old Card Value', dodge=False, **marker_style(2), chance=0.5,
             xlabel='Encoding-Retrieval Delay', ylabel='P(Choose Old Card)'),
        dict(data=old_trials, plot_fn=plot_subject_lines, title='Old Trial Choices', fname='old_choices.png',
             x='old_value', y='old_chosen', color=MEM_PALETTES['blue']['mid'], native_scale=True, **value_ticks,
             xlabel='Old Card Value', ylabel='P(Choose Old Card)'),
        dict(mem_choices, fname='old_choices_mem.png'),
        # dodged alternatives: high memorability rightmost / leftmost at each value (legend stays low -> high)
        dict(mem_choices, fname='old_choices_mem_dodgeright.png', dodge=True),
        dict(mem_choices, fname='old_choices_mem_dodgeleft.png', dodge=True, hue_order=MEM_ORDER[::-1],
             hue_labels=MEM_LABELS[::-1], legend_reverse=True),
        # incorrect in gray, correct in purple, light -> dark by memorability
        dict(data=old_trials, plot_fn=plot_colored_means, title='Experiment 1A - RTs', fname='old_rt_mem.png',
             ylim=(None, None), ytick_step=0.05, x='memorability_bin', y='rt_sec', order=MEM_ORDER, palette=None,
             split_by='optimal_old_choice', **correct_split, **marker_style(2), xticks=[0, 1, 2], xticklabels=MEM_LABELS,
             legend_opts=dict(allow_overlap=True), xlabel='Memorability Bin', ylabel='RT (s)'),
    ]


def plot_old_rt_all(sources, figures_dir, fname='old_rt_all.png', panel_size=(3.5, 4.8), axes_size=None):
    """Old-trial RT three ways, a row of 3 panels per source (exp, label): by correct vs. incorrect choice, by
    whether the old card was chosen, and both at once (incorrect in gray, correct in purple). Included participants."""
    mem_kws = dict(y='rt_sec', hue='memorability_bin', hue_order=MEM_ORDER, palette=MEM_PALETTES['purple'],
                   **marker_style(3), extra_by=['old_value'], dodge=True, errorbar=FIG_DEFAULTS['errorbar'],
                   hue_labels=MEM_LABELS, legend_title='Memorability Bin', ytick_step=0.05, ylabel='RT (s)')
    both_kws = dict(y='rt_sec', hue='correct_mem', hue_order=[f'{c}_{m}' for c in ('incorrect', 'correct') for m in MEM_ORDER],
                    palette={**{f'incorrect_{m}': GRAY_PALETTE[m] for m in MEM_ORDER},
                             **{f'correct_{m}': MEM_PALETTES['purple'][m] for m in MEM_ORDER}},
                    **marker_style(3), dodge=0.5, errorbar=FIG_DEFAULTS['errorbar'], legend_title='Memorability Bin',
                    hue_labels=[f'{l}, {c}' for c in ('Incorrect', 'Correct') for l in MEM_LABELS], ytick_step=0.05)
    w, h = panel_size
    fig = plt.figure(figsize=(3 * w, len(sources) * h), layout='constrained')
    rows = np.atleast_1d(fig.subfigures(len(sources), 1))
    for row, (exp, label) in zip(rows, sources):
        rt_df = exp1_old_trials(exp)
        rt_df = rt_df[rt_df['participant_id'].isin(exp.included_pids)].copy()
        rt_df['correct_mem'] = (rt_df['optimal_old_choice'].map({0: 'incorrect', 1: 'correct'}) + '_'
                                + rt_df['memorability_bin'].astype(str))
        axes = row.subplots(1, 3, sharey=True)
        n = plot_hue_means(axes[0], rt_df, x='optimal_old_choice', **mem_kws, **correct_ticks, xlabel='Choice')
        plot_hue_means(axes[1], rt_df, x='old_chosen', **mem_kws, **no_yes, xlabel='Chose Old Card')
        plot_hue_means(axes[2], rt_df, x='old_chosen', **both_kws, **no_yes, xlabel='Chose Old Card')
        for ax, panel_title in zip(axes, ['Correct vs. Incorrect', 'Old Card Chosen', 'Both']):
            ax.set_title(panel_title)
            sns.move_legend(ax, 'lower left', fontsize=8, title_fontsize=8)
        axes[1].get_legend().remove()  # same key as the first panel
        if label is not None or USE_PILOT:
            row.suptitle(f'{label} (n={n})' if label else f'n={n}', fontsize=plt.rcParams['axes.titlesize'])
    fig.suptitle('Old Trial RT')
    finish_fig(figures_dir, fname, tight=False, axes_size=axes_size)  # constrained layout already placed everything


plot_figures(exp1_figures, exp1, exp1_pilot)
plot_old_rt_all([(exp1_pilot, 'Pilot'), (exp1, 'Replication')] if COMPARE_PILOT else [(exp1, None)], exp1.figures_dir,
                axes_size=_standalone_axes_size(exp1_figures, exp1, exp1_pilot, 'old_choices_mem.png'))

# Experiment 2 (Direct Recall)

The direct task has no old/new choice on repeated-card trials -- it has two separate memory tests instead:
recognition (did you pick the seen-before card?) and a value report (what was it worth?). Recognition accuracy
(analogous to `optimal_old_choice` elsewhere) defines the per-subject significance / inclusion set.

In [ ]:
DIRECT_SETUP = dict(label='direct', perf_query='is_recognition_trial == True', perf_col='recognition_correct',
                    perf_label='Recognition Accuracy', min_trials=100)
exp2 = setup_experiment(**DIRECT_SETUP)
exp2_pilot = setup_experiment(**DIRECT_SETUP, data_root=PILOT_ROOT) if COMPARE_PILOT else None
# print_bonuses(exp2.df)

In [ ]:
def load_direct_trials(exp):
    """Recognition and value-report trials, with their outcome columns made numeric."""
    recog_df = exp.df.query('is_recognition_trial == True').copy()
    recog_df['recognition_correct'] = pd.to_numeric(recog_df['recognition_correct'], errors='coerce')
    recog_df['choice_missed'] = pd.to_numeric(recog_df['choice_missed'], errors='coerce')

    value_df = exp.df.query('is_value_test_trial == True').copy()
    value_df['value_test_correct'] = pd.to_numeric(value_df['value_test_correct'], errors='coerce')
    value_df['value_test_missed'] = pd.to_numeric(value_df['value_test_missed'], errors='coerce')
    value_df['old_value'] = pd.to_numeric(value_df['old_value'], errors='coerce')
    return recog_df, value_df


# Note: a subset of sessions (mostly Windows/Chrome) have a bug where keyboard responses on the value-report screen
# intermittently fail to register, while recognition-phase responses stay normal. The preregistered > 20% miss-rate
# exclusion (across all response trials) is applied in setup_experiment.

In [ ]:
mem_ticks = dict(xticks=[0, 1, 2], xticklabels=MEM_LABELS, xlabel='Memorability Bin')
hit_miss_panels = dict(panel_by='recognition_hit', panel_order=[1.0, 0.0], panel_titles=['Recognition Hits', 'Recognition Misses'])


def exp2_figures(exp):
    recog_df, value_df = load_direct_trials(exp)
    recog_valid = recog_df.dropna(subset=['recognition_correct', 'rt']).astype({'recognition_correct': int})
    recog_valid['rt_sec'] = recog_valid['rt'] / 1000
    # each value report follows its recognition trial (same trial_number); hit = recognized correctly
    value_df = value_df.merge(recog_df[['participant_id', 'trial_number', 'recognition_correct']]
                              .rename(columns={'recognition_correct': 'recognition_hit'}),
                              on=['participant_id', 'trial_number'], how='left')
    value_valid = value_df.dropna(subset=['value_test_correct', 'rt']).astype({'value_test_correct': int})
    value_valid['rt_sec'] = value_valid['rt'] / 1000
    # recognition and value-recall accuracy stacked, one row per trial, for plotting on the same axes;
    # value recall only on trials whose card was correctly recognized (recognition hits)
    value_hits = value_df[value_df['recognition_hit'] == 1]
    both_tests = pd.concat([recog_df.assign(test='recognition', accuracy=recog_df['recognition_correct']),
                            value_hits.assign(test='value', accuracy=value_hits['value_test_correct'])])

    stacked = dict(data=both_tests, plot_fn=plot_colored_means, title='Experiment 1B',
                   ylim=(None, 0.925), x='memorability_bin', y='accuracy', order=MEM_ORDER, palette=None, split_by='test',
                   split_order=['recognition', 'value'], split_labels=['Recognition', 'Value Recall'],
                   split_palettes=[MEM_PALETTES['blue'], MEM_PALETTES['blue']],
                   split_line_colors=['black'] * 2, split_markers=['s', 'o'], dodge=0,
                   legend_opts=dict(loc='lower right', fontsize=10, allow_overlap=True),
                   **marker_style(2), chance=0.5, **mem_ticks, ylabel='Accuracy')

    return [
        # recognition (squares) and value recall (circles) on the same axes, light -> dark blue by memorability
        dict(stacked, fname='recognition_recall_stacked.png'),
        dict(data=recog_df, plot_fn=plot_colored_means, title='Recognition', fname='recognition.png',
             ylim=(None, 1.0), x='memorability_bin', y='recognition_correct', order=MEM_ORDER,
             palette=MEM_PALETTES['blue'], chance=0.5, **mem_ticks, ylabel='Accuracy'),
        dict(data=recog_valid, plot_fn=plot_colored_means, title='Recognition RTs', panel_title='Recognition',
             fname='recognition_rt.png', ylim=(None, 1.1), ytick_step=0.05, x='memorability_bin', y='rt_sec',
             order=MEM_ORDER, palette=None, split_by='recognition_correct', **correct_split, **marker_style(2),
             legend_opts=dict(allow_overlap=True), **mem_ticks, ylabel='RT (s)'),
        # value recall on recognition hits only (also the right panel of recognition_recall.png)
        dict(data=value_hits, plot_fn=plot_colored_means, title='Value Recall', fname='recall_value.png',
             ylim=(None, 0.7), yticks=[0.5, 0.55, 0.6, 0.65, 0.7], x='memorability_bin', y='value_test_correct', order=MEM_ORDER,
             palette=MEM_PALETTES['blue'], chance=0.5, **mem_ticks, ylabel='Accuracy'),
        dict(data=value_valid, plot_fn=plot_colored_means, title='Value Recall RTs', panel_title='Value Recall',
             fname='recall_value_rt.png', ylim=(None, 0.75), ytick_step=0.05, x='memorability_bin', y='rt_sec',
             order=MEM_ORDER, palette=None, split_by='value_test_correct', **correct_split, **marker_style(2),
             legend_opts=dict(allow_overlap=True), **mem_ticks, ylabel='RT (s)'),
        # value recall split by whether the card was recognized (hit) or not (miss)
        dict(data=value_df, plot_fn=plot_colored_means, title='Experiment 1B: Value Recall by Recognition Performance',
             panel_groups=['Recognition Hits', 'Recognition Misses'], fname='recall_value_by_recognition.png',
             ylim=(None, 0.7), **hit_miss_panels, x='memorability_bin', y='value_test_correct', order=MEM_ORDER,
             palette=MEM_PALETTES['blue'], chance=0.5, **mem_ticks, ylabel='Value Recall Accuracy'),
    ]

In [ ]:
plot_figures(exp2_figures, exp2, exp2_pilot)
plot_side_by_side(exp2_figures, exp2, exp2_pilot, ['recognition.png', 'recall_value.png'], 'Experiment 1B', 'recognition_recall.png')
plot_side_by_side(exp2_figures, exp2, exp2_pilot, ['recognition_rt.png', 'recall_value_rt.png'], 'Experiment 1B - RTs',
                  'recognition_recall_rt.png')

# Experiment 3 (Matched-Memorability)

In [ ]:
exp3 = setup_experiment('matched_memorability', min_trials=70)
exp3_pilot = setup_experiment('matched_memorability', min_trials=70, data_root=PILOT_ROOT) if COMPARE_PILOT else None
# print_bonuses(exp3.df)

In [ ]:
MATCHED_MEM_ORDER = ['low', 'high']

matched_ticks = dict(xticks=[0, 1], xticklabels=['Low', 'High'], xlim=(-0.5, 1.5), xlabel='Memorability Bin')

def exp3_figures(exp):
    old_matched = exp.df.query('old_trial == 1').copy()
    old_matched['optimal_old_choice'] = pd.to_numeric(old_matched['optimal_old_choice'], errors='coerce')
    old_matched['rt_sec'] = old_matched['rt'] / 1000
    old_matched = old_matched.dropna(subset=['optimal_old_choice'])
    old_matched['chose_right'] = old_matched['chosen_side'].map({'left': 0.0, 'right': 1.0})
    blue = MEM_PALETTES['blue']
    return [
        # by which card is worth more, one line per shared memorability bin (light / dark blue)
        dict(data=old_matched, plot_fn=plot_colored_means, title='Experiment 2A (P1)', fname='old_choices.png',
             axes_size=SHORT_AXES_SIZE, legend_opts=dict(loc='lower right', fontsize=10, title_fontsize=10), x='right_value', y='chose_right',
             order=[0.0, 1.0], palette=None, dodge=0, split_by='memorability_bin', split_order=MATCHED_MEM_ORDER,
             split_labels=['Low', 'High'], legend_title='Shared Memorability',
             split_palettes=[dict.fromkeys([0.0, 1.0], blue[m]) for m in MATCHED_MEM_ORDER],
             split_line_colors=[blue[m] for m in MATCHED_MEM_ORDER], **marker_style(2), chance=0.5, **side_ticks,
             xlabel='Card Values', ylabel='P(Choose Right Card)'),
        dict(data=old_matched, plot_fn=plot_colored_means, title='Experiment 2A (P1)', fname='old_choices_v2.png', ylim=(None, 0.8),
             x='memorability_bin', y='optimal_old_choice', order=MATCHED_MEM_ORDER, palette=dict.fromkeys(MATCHED_MEM_ORDER, 'black'),
             chance=0.5, **matched_ticks, ylabel='P(Choose Higher Value)'),
        dict(data=old_matched, plot_fn=plot_colored_means, title='Experiment 2A - RTs', fname='old_rt.png', ylim=(1.0, None), ytick_step=0.05,
             legend_opts=dict(loc='lower left', allow_overlap=True), x='memorability_bin', y='rt_sec',
             order=MATCHED_MEM_ORDER, palette=None, split_by='optimal_old_choice', **correct_split, **marker_style(2), **matched_ticks,
             ylabel='RT (s)'),
    ]


plot_figures(exp3_figures, exp3, exp3_pilot)

# Experiment 4 (Mixed-Memorability)

df.ret_type:
- 1 -> high=0, low=1
- 2 -> high=1, low=0
- 3 -> high=0, low=0
- 4 -> high=1, low=1

In [ ]:
# significance is on optimal_old_choice, which is only defined on the uneven (ret_type 1/2) trials
exp4 = setup_experiment('mixed_memorability', min_trials=70)
exp4_pilot = setup_experiment('mixed_memorability', min_trials=70, data_root=PILOT_ROOT) if COMPARE_PILOT else None
# print_bonuses(exp4.df)

In [ ]:
# Human-readable labels & display order
RT_LABELS = {1: 'High=\\$0\nLow=\\$1', 2: 'High=\\$1\nLow=\\$0', 3: 'Both=\\$0', 4: 'Both=\\$1'}
UNEVEN_ORDER = [RT_LABELS[k] for k in (1, 2)]
EVEN_ORDER   = [RT_LABELS[k] for k in (3, 4)]

value_ticks = dict(xticks=[0, 1], xticklabels=['$0', '$1'], xlim=(-0.5, 1.5))
uneven_palette = dict(zip(UNEVEN_ORDER, VALUE_PALETTE.values()))
even_palette = dict(zip(EVEN_ORDER, VALUE_PALETTE.values()))
uneven_ticks = dict(xticks=[0, 1], xticklabels=['High=\\$0\nLow=\\$1', 'High=\\$1\nLow=\\$0'], xlim=(-0.5, 1.5))


def exp4_figures(exp):
    old_mixed = exp.df.query('old_trial == 1').copy()
    old_mixed['optimal_old_choice'] = pd.to_numeric(old_mixed['optimal_old_choice'], errors='coerce')
    # Which memorability bin did the participant effectively choose?
    old_mixed['chose_high_mem'] = (
        ((old_mixed['chosen_side'] == 'left')  & (old_mixed['left_mem_bin']  == 'high')) |
        ((old_mixed['chosen_side'] == 'right') & (old_mixed['right_mem_bin'] == 'high'))
    ).astype(float)
    old_mixed['rt_sec'] = old_mixed['rt'] / 1000
    old_mixed['ret_type_label'] = old_mixed['ret_type'].map(RT_LABELS)
    old_mixed['chose_right'] = old_mixed['chosen_side'].map({'left': 0.0, 'right': 1.0})
    # which card is more memorable ('left' / 'right')
    old_mixed['mem_side'] = np.where(old_mixed['right_mem_bin'] == 'high', 'right', 'left')
    old_mixed['shared_value'] = old_mixed['ret_type'].map({3: 0.0, 4: 1.0})
    uneven_trials = old_mixed.dropna(subset=['optimal_old_choice'])
    even_trials = old_mixed.query('ret_type in [3, 4]')

    matched_black = dict(zip(EVEN_ORDER, ['black'] * 2))
    mixed_black = dict(zip(UNEVEN_ORDER, ['black'] * 2))
    purple = MEM_PALETTES['purple']
    sides = ['left', 'right']

    return [
        # by which card is more memorable, one line per shared value (light / dark orange)
        dict(data=even_trials, plot_fn=plot_colored_means, title='Experiment 2B (P2)', fname='matched_value_choices.png',
             axes_size=SHORT_AXES_SIZE, ytick_step=0.05, x='mem_side', y='chose_right', order=sides, palette=None, dodge=0,
             split_by='shared_value', split_order=[0.0, 1.0], split_labels=['\\$0', '\\$1'], legend_title='Shared Value',
             split_palettes=[dict.fromkeys(sides, VALUE_PALETTE[v]) for v in (0.0, 1.0)],
             split_line_colors=[VALUE_PALETTE[0.0], VALUE_PALETTE[1.0]], **marker_style(2), chance=0.5, **side_ticks,
             xlabel='Card Memorabilities', ylabel='P(Choose Right Card)'),
        dict(data=even_trials, plot_fn=plot_colored_means, title='Experiment 2B (P2)',
             panel_title='Matched-Value Trials', fname='matched_value_choices_v2.png', ylim=(0.35, 0.65), x='ret_type_label', y='chose_high_mem',
             order=EVEN_ORDER, palette=matched_black, line_color='black', chance=0.5, **value_ticks,
             xlabel='Shared Card Value', ylabel='P(Choose Memorable)'),
        # chosen card's memorability on the x axis (light / dark purple), one black line per shared value: $0 open, $1 filled
        dict(data=even_trials, plot_fn=plot_colored_means, title='Experiment 2B - RTs\n(matched-value trials)',
             panel_title='Matched-Value Trials', fname='matched_value_rt.png', ylim=(1.0, None), ytick_step=0.05,
             x='chose_high_mem', y='rt_sec', order=[0.0, 1.0], palette=None, split_by='ret_type_label',
             split_order=EVEN_ORDER, split_labels=['Both = \\$0', 'Both = \\$1'],
             split_palettes=[{0.0: purple['low'], 1.0: purple['high']}] * 2,
             split_line_colors=['black'] * 2, split_open=[True, False], legend_colors=[purple['mid']] * 2,
             **marker_style(2),
             xticks=[0, 1], xticklabels=['Low', 'High'], xlim=(-0.5, 1.5), legend_opts=dict(fontsize=10, grow=1.02),
             xlabel='Chosen Card Memorability', ylabel='RT (s)'),
        dict(data=uneven_trials, plot_fn=plot_colored_means, title='Experiment 2B\n(mixed-value trials)',
             panel_title='Mixed-Value Trials', fname='mixed_value_choices.png', ylim=(None, 0.9),
             x='ret_type_label', y='optimal_old_choice', order=UNEVEN_ORDER, palette=mixed_black, line_color='black',
             chance=0.5, **uneven_ticks, ylabel='P(Choose Higher Value)'),
        # incorrect in gray, correct in purple (middle shades)
        dict(data=uneven_trials, plot_fn=plot_colored_means, title='Experiment 2B - RTs\n(mixed-value trials)',
             panel_title='Mixed-Value Trials', fname='mixed_value_rt.png', ylim=(1.0, None),
             x='ret_type_label', y='rt_sec', order=UNEVEN_ORDER, legend_opts=dict(loc='lower left', allow_overlap=True),
             palette=None, split_by='optimal_old_choice', split_order=[0.0, 1.0],
             split_labels=['Incorrect', 'Correct'],
             split_palettes=[dict(zip(UNEVEN_ORDER, [GRAY_PALETTE['mid']] * 2)), dict(zip(UNEVEN_ORDER, [purple['mid']] * 2))],
             split_line_colors=[GRAY_PALETTE['mid'], purple['mid']], **marker_style(2), **uneven_ticks,
             ytick_step=0.05, ylabel='RT (s)'),
    ]


plot_figures(exp4_figures, exp4, exp4_pilot)
plot_side_by_side(exp4_figures, exp4, exp4_pilot, ['matched_value_choices_v2.png', 'mixed_value_choices.png'],
                  'Experiment 2B', 'choices.png', share_ylabel=False, legends='all')
plot_side_by_side(exp4_figures, exp4, exp4_pilot, ['matched_value_rt.png', 'mixed_value_rt.png'], 'Experiment 2B - RTs',
                  'rt.png', legends='all')

# Experiment 5 - Food Choice

In [ ]:
import statsmodels.api as sm


def food_delta_value_fit(g):
    """Li et al. (2022) value check for one participant: logistic regression of choosing the right item on
    delta value (z-scored rating of right item - left item). Returns (slope, Wald p). Perfectly separated
    data (every choice follows the ratings) has no finite MLE and is returned as (inf, 0.0)."""
    x, y = g['delta_value'].to_numpy(), g['chose_right'].to_numpy()
    if ((x > 0) & (y == 0)).sum() + ((x < 0) & (y == 1)).sum() == 0 and (x != 0).any():
        return np.inf, 0.0
    try:
        fit = sm.Logit(y, sm.add_constant(x)).fit(disp=0)
    except Exception:  # e.g. every choice on one side -> no estimable slope, counts as non-significant
        return np.nan, np.nan
    return fit.params[1], fit.pvalues[1]


FOOD_CHOICE_TRIALS = {'gap23': 92, 'gap8': 96}  # choice trials per version (fewer counts as incomplete)


def setup_food_experiment(gap, data_root=DATA_ROOT):
    """Load the food-choice data and run its exclusions (same criteria as scripts/count_participants.py).

    gap: 'gap23' (preregistered, 92 foods) or 'gap8' (96 foods); figures go in figures/food_choice/<gap>/.

    Excluded: suspected AI, any failed attention check, fewer than FOOD_CHOICE_TRIALS[gap] choice trials, > 20% missed choices (preregistered),
    plus Li et al.'s (2022) Exp 1B criteria: P(choose right) more than 2 SD from the mean across all participants;
    RT < 300 ms on more than half of trials; a negative or non-significant (p >= .05) effect of delta value on choice
    in a per-participant logistic regression. The Li criteria use responded choice trials; the value regression also
    drops RT < 300 ms trials, as Li et al.'s analyses do."""
    label = f'food_choice/{gap}'
    print(f'===== {label} =====')
    df, figures_dir = load_data(label, data_root)
    ai_pids = find_ai_pids(df)
    failed_pids = find_failed_attention_pids(df)
    incomplete_pids = find_incomplete_pids(df, 'is_choice_trial == True', FOOD_CHOICE_TRIALS[gap])
    high_miss_pids = find_high_miss_pids(df)
    print(f'{len(high_miss_pids)} participants with > 20% missed responses')

    choice_df = df.query('is_choice_trial == True').copy()
    missed = choice_df['choice_missed'].map(lambda v: str(v).lower() in ('true', '1', '1.0'))
    responded = choice_df[~missed].copy()
    responded['chose_right'] = pd.to_numeric(responded['chose_right'])
    responded['rt'] = pd.to_numeric(responded['rt'])

    p_right = responded.groupby('participant_id')['chose_right'].mean()
    side_bias_pids = p_right[(p_right - p_right.mean()).abs() > 2 * p_right.std()].index.tolist()
    print(f'{len(side_bias_pids)} participants with P(choose right) > 2 SD from the mean ({p_right.mean():.3f} +/- {2 * p_right.std():.3f})')
    fast_rate = (responded['rt'] < 300).groupby(responded['participant_id']).mean()
    fast_pids = fast_rate[fast_rate > 0.5].index.tolist()
    print(f'{len(fast_pids)} participants with RT < 300 ms on more than half of trials')

    ratings = df.query('is_rating_trial == True')
    z = ratings.groupby('participant_id')['rating'].transform(lambda r: (r - r.mean()) / r.std())
    z_value = dict(zip(zip(ratings['participant_id'], ratings['image_name']), z))
    fit_df = responded.query('rt >= 300').copy()
    fit_df['delta_value'] = [z_value[(p, r)] - z_value[(p, l)] for p, r, l in
                             zip(fit_df['participant_id'], fit_df['right_image_name'], fit_df['left_image_name'])]
    value_fits = pd.DataFrame([(pid, *food_delta_value_fit(g)) for pid, g in fit_df.groupby('participant_id')],
                              columns=['participant_id', 'slope', 'pval'])
    value_pids = value_fits.query('not (slope > 0 and pval < 0.05)')['participant_id'].tolist()
    value_pids += [pid for pid in choice_df['participant_id'].unique() if pid not in set(value_fits['participant_id'])]
    print(f'{len(value_pids)} participants with a negative or non-significant effect of delta value on choice')

    exp = SimpleNamespace(label=label, source='Replication', df=df, figures_dir=figures_dir, ai_pids=ai_pids, failed_pids=failed_pids,
                          incomplete_pids=incomplete_pids, high_miss_pids=high_miss_pids, side_bias_pids=side_bias_pids,
                          fast_pids=fast_pids, value_pids=value_pids, value_fits=value_fits,
                          sig_pids=df['participant_id'].unique().tolist(), excluded_pids=set())
    exclude(exp, ai_pids + failed_pids + incomplete_pids + high_miss_pids + side_bias_pids + fast_pids + value_pids)
    print(f'Included n = {len(exp.included_pids)}')
    return exp


# no pilot for this task: runs on the replication data only, each version once some has been collected
exp5 = {gap: setup_food_experiment(gap) for gap in FOOD_CHOICE_TRIALS
        if not USE_PILOT and os.path.exists(f'{DATA_ROOT}/data/food_choice_data_{gap}.csv')}

In [ ]:
### FIGURES: USE_PILOT draws Li et al.'s (2022) Exp 1B data (saved under their pilot folder), COMPARE_PILOT draws theirs
# next to our replication, and otherwise just the replication.
# All figures use included participants' responded choices with RT >= 300 ms (as in Li et al.'s analyses).
import re

LI_FIGURES_DIR = '../pilot/li_bainbridge_bakkour_2022/figures'
SUM_VALUE_PALETTE = {0: '#fdd0a2', 1: '#fd8d3c', 2: '#a63603'}   # overall value terciles, light -> dark
MEM_PAIR_PALETTE = {'low_low': MEM_PALETTES['blue']['low'], 'high_high': MEM_PALETTES['blue']['high']}
TERCILE_LABELS = ['Low', 'Mid', 'High']
bin_ticks = lambda n: dict(xticks=list(range(n)), xticklabels={3: TERCILE_LABELS, 4: ['Low', '', '', 'High'],
                                                               5: ['Low', '', 'Mid', '', 'High']}[n])


def load_li_food_data():
    """Li et al.'s (2022) Exp 1B data (already post-exclusion) in our food-choice format. Their pairs are value-adjacent
    (similar) or 69 value ranks apart (distant); memorability groups are our fixed tercile split from the task file
    (the middle tercile, which our task does not use, has no group)."""
    task_js = open('../tasks/food_choice/replication_gap23/memorability_choice_image.js').read()
    mem_groups = dict(re.findall(r'image_name: "([^"]+)", memorability: [-\d.]+, mem_group: "(\w+)"', task_js))
    li = pd.read_csv('../pilot/li_bainbridge_bakkour_2022/data/Exp1B.csv', low_memory=False)
    li['participant_id'] = 'li_' + li['ID'].astype(str)
    image = lambda urls: urls.str.split('/').str[-1]

    ratings = li.query('ttype == "rating_task"').copy()
    ratings['is_rating_trial'] = True
    ratings['image_name'] = image(ratings['image'])
    ratings['rating'] = pd.to_numeric(ratings['response'])
    rating_of = ratings.set_index(['participant_id', 'image_name'])['rating']
    rank_of = ratings.assign(rank=ratings.groupby('participant_id')['rating'].rank()).set_index(['participant_id', 'image_name'])['rank']

    choices = li.query('ttype == "choice_task"').copy()
    choices['is_choice_trial'] = True
    choices['rt'] = pd.to_numeric(choices['rt'], errors='coerce')
    choices['choice_missed'] = choices['rt'].isna()
    choices['chose_right'] = pd.to_numeric(choices['key_press'], errors='coerce').map({75: 1, 74: 0})
    for side in ('left', 'right'):
        choices[f'{side}_image_name'] = image(choices[f'stim_{side}'])
        keys = list(zip(choices['participant_id'], choices[f'{side}_image_name']))
        choices[f'{side}_memorability'] = pd.to_numeric(choices[f'mem_{side}'])
        choices[f'{side}_mem_group'] = choices[f'{side}_image_name'].map(mem_groups)
        choices[f'{side}_value'] = rating_of.loc[keys].values
        choices[f'{side}_rank'] = rank_of.loc[keys].values
    choices['pair_type'] = np.where((choices['left_rank'] - choices['right_rank']).abs() < 35, 'similar', 'distant')
    left, right = choices['left_mem_group'], choices['right_mem_group']
    choices['mem_pair_type'] = np.select([left.isna() | right.isna(), left == right], ['with_mid', left + '_' + right], 'high_low')
    df = pd.concat([ratings, choices], ignore_index=True)
    return SimpleNamespace(label='li', df=df, included_pids=df['participant_id'].unique().tolist())


def equal_bins(x, q):
    return pd.qcut(x.rank(method='first'), q, labels=False)  # equal-sized bins, ties split


def food_trials(exp):
    """Included participants' responded choices with RT >= 300 ms, with within-participant z-scored values,
    value and memorability differences (left minus right, and Li's right minus left) and sums, and choice outcomes."""
    df = exp.df[exp.df['participant_id'].isin(exp.included_pids)]
    ratings = df.query('is_rating_trial == True')
    z = ratings.groupby('participant_id')['rating'].transform(lambda r: (r - r.mean()) / r.std())
    z_of = dict(zip(zip(ratings['participant_id'], ratings['image_name']), z))
    t = df.query('is_choice_trial == True').copy()
    t = t[~t['choice_missed'].map(lambda v: str(v).lower() in ('true', '1', '1.0'))]
    t['rt'] = pd.to_numeric(t['rt'])
    t = t[t['rt'] >= 300].copy()
    t['rt_sec'] = t['rt'] / 1000
    t['chose_right'] = pd.to_numeric(t['chose_right'])
    for side in ('left', 'right'):
        t[f'{side}_z'] = [z_of[key] for key in zip(t['participant_id'], t[f'{side}_image_name'])]
        t[f'{side}_memorability'] = pd.to_numeric(t[f'{side}_memorability'])
    t['delta_value'] = t['right_z'] - t['left_z']
    t['delta_mem'] = t['right_memorability'] - t['left_memorability']
    t['sum_value'] = t['right_z'] + t['left_z']
    chose = lambda delta: np.where(delta == 0, np.nan, (t['chose_right'] == (delta > 0)).astype(float))
    t['chose_higher_value'] = chose(t['delta_value'])  # NaN on ties: neither item is higher
    t['chose_more_mem'] = chose(t['delta_mem'])
    by_pid = t.groupby('participant_id')
    t['low_abs_dmem'] = by_pid['delta_mem'].transform(lambda x: x.abs() <= x.abs().median())      # Li's median splits
    t['low_abs_dvalue'] = by_pid['delta_value'].transform(lambda x: x.abs() <= x.abs().median())
    return t


def fig3_panels(t):
    """Li et al.'s Figure 3: value on low-|dmem| trials, memorability on low-|dvalue| trials (median splits), in
    5 equal-sized |delta| bins. Ties (neither item higher) are dropped; RT panels keep only higher-value choices."""
    value = t[t['low_abs_dmem'] & (t['delta_value'] != 0)].copy()
    value['bin'] = equal_bins(value['delta_value'].abs(), 5)
    mem = t[t['low_abs_dvalue'] & (t['delta_mem'] != 0)].copy()
    mem['bin'] = equal_bins(mem['delta_mem'].abs(), 5)
    value_color, mem_color, rt_color = VALUE_PALETTE[1.0], MEM_PALETTES['blue']['mid'], MEM_PALETTES['purple']['mid']
    rt = dict(y='rt_sec', color=rt_color, chance=None, ytick_step=0.2, ylabel='RT (s)')
    return [
        (value, 'Value and Choice\n(Low |ΔMemorability| Trials)',
         dict(y='chose_higher_value', color=value_color, xlabel='|ΔValue|', ylabel='P(Choose Higher Value)')),
        (mem, 'Memorability and Choice\n(Low |ΔValue| Trials)',
         dict(y='chose_more_mem', color=mem_color, xlabel='|ΔMemorability|', ylabel='P(Choose More Memorable)')),
        (value.query('chose_higher_value == 1'), 'Value and RT\n(Low |ΔMemorability| Trials,\nHigher-Value Choices)', dict(**rt, xlabel='|ΔValue|')),
        (mem.query('chose_higher_value == 1'), 'Memorability and RT\n(Low |ΔValue| Trials,\nHigher-Value Choices)', dict(**rt, xlabel='|ΔMemorability|')),
    ]


def plot_fig3(sources, figures_dir, fname='li_replication.png', panel_size=(3.8, 4.2)):
    """Li et al.'s Figure 3 as one figure: a row of 4 panels per source (exp, label). n is each source's included
    sample (a panel can have fewer: e.g. someone with no higher-value choices on low-|dvalue| trials)."""
    w, h = panel_size
    fig = plt.figure(figsize=(4 * w, len(sources) * (h + 0.8)), layout='constrained')
    rows = np.atleast_1d(fig.subfigures(len(sources), 1))
    for row, (exp, label) in zip(rows, sources):
        axes = row.subplots(1, 4)
        trials = food_trials(exp)
        n = trials['participant_id'].nunique()
        for ax, (data, title, kws) in zip(axes, fig3_panels(trials)):
            plot_subject_lines(ax, data, x='bin', errorbar=FIG_DEFAULTS['errorbar'], **marker_style(1), **bin_ticks(5), **kws)
            ax.set_title(title, fontsize=13)
        row.suptitle(label if label and 'n=' in label else f'{label} (n={n})' if label
                     else f'Replication of Li, Bainbridge, & Bakkour (2022) Results (n={n})')
    if save:
        fig.savefig(os.path.join(figures_dir, fname), bbox_inches='tight')
    plt.show()


def food_figures(exp):
    t = food_trials(exp)
    # similar-value trials, as in David Halpern's li_reanalysis_1B.R: the lower half of each participant's trials by
    # |delta value| (rank <= median rank, ties in trial order); delta memorability (right - left) in 3 bins at the
    # normal terciles (.33, .66) of delta / SD over these trials, so bins cover the same range for everyone and the
    # middle one is centered on 0; overall value terciles pooled over participants; participants missing any of the
    # 9 bin x tercile cells are dropped
    by_abs = t.assign(abs_dv=t['delta_value'].abs()).sort_values(['participant_id', 'abs_dv'], kind='stable')
    rank = by_abs.groupby('participant_id').cumcount() + 1
    n_trials = by_abs.groupby('participant_id')['participant_id'].transform('size')
    similar = by_abs[rank <= (n_trials + 1) / 2].copy()
    similar['dmem_bin'] = pd.cut(similar['delta_mem'] / similar['delta_mem'].std(), stats.norm.ppf([0, .33, .66, 1]), labels=False)
    similar['sum_value_bin'] = equal_bins(similar['sum_value'], 3)
    n_cells = similar.groupby('participant_id').apply(lambda g: len(g[['dmem_bin', 'sum_value_bin']].drop_duplicates()), include_groups=False)
    similar = similar[similar['participant_id'].map(n_cells) == 9]
    # distant-value trials between two high- or two low-memorability items, by whether the right item is the higher-valued
    # or by delta value (right - left) in 3 bins at the normal terciles of delta / SD, as for delta memorability above
    # (distant deltas are bimodal, so the middle bin holds few trials and only some participants)
    distant = t.query('pair_type == "distant" and mem_pair_type in ["low_low", "high_high"] and delta_value != 0').copy()
    distant['right_higher'] = (distant['delta_value'] > 0).astype(int)
    distant['dvalue_bin'] = pd.cut(distant['delta_value'] / distant['delta_value'].std(), stats.norm.ppf(np.linspace(0, 1, 4)), labels=False)
    distant['choice_mem'] = distant['chose_higher_value'].map({0: 'lower', 1: 'higher'}) + '_' + distant['mem_pair_type']

    legend_opts = dict(fontsize=9, title_fontsize=9, allow_overlap=True)  # keep the figure size; legend where it overlaps least
    # Li-only figures (one panel): extra room above the data for an upper-left legend
    sum_value_hue = dict(plot_fn=plot_hue_means, **marker_style(3), figsize=(4.4, 4.2),
                         legend_opts=dict(legend_opts, loc='upper left') if USE_PILOT else legend_opts,
                         x='dmem_bin', hue='sum_value_bin', hue_order=[0, 1, 2], palette=SUM_VALUE_PALETTE,
                         hue_labels=TERCILE_LABELS, legend_title='Overall Value', **bin_ticks(3),
                         xlabel='ΔMemorability (Right - Left)')
    mem_hue = dict(plot_fn=plot_hue_means, **marker_style(2), figsize=(4.4, 4.2), legend_opts=legend_opts,
                   hue='mem_pair_type', hue_order=list(MEM_PAIR_PALETTE), palette=MEM_PAIR_PALETTE,
                   hue_labels=['Both low', 'Both high'], legend_title='Memorability Bin')
    # RT: lower-value choices in gray, higher-value choices in blue; both light -> dark for both-low -> both-high memorability
    choice_mem_hue = dict(plot_fn=plot_hue_means, **marker_style(3), figsize=(5, 4.4),
                          legend_opts=dict(fontsize=9, title_fontsize=9, grow=1.04),  # 4 entries: grow until it clears the data
                          hue='choice_mem', hue_order=['lower_low_low', 'lower_high_high', 'higher_low_low', 'higher_high_high'],
                          palette={'lower_low_low': GRAY_PALETTE['low'], 'lower_high_high': GRAY_PALETTE['high'],
                                   'higher_low_low': MEM_PAIR_PALETTE['low_low'], 'higher_high_high': MEM_PAIR_PALETTE['high_high']},
                          hue_labels=['Both low, lower-value choice', 'Both high, lower-value choice',
                                      'Both low, higher-value choice', 'Both high, higher-value choice'],
                          legend_title='Memorability Bin', dodge=0.4, y='rt_sec', ytick_step=0.1, ylabel='RT (s)')
    right_value_ticks = dict(x='right_higher', order=[0, 1], xticks=[0, 1], xticklabels=['Lower', 'Higher'], xlabel='Right Item Value')
    delta_value_ticks = dict(x='dvalue_bin', order=[0, 1, 2], **bin_ticks(3), xlabel='ΔValue (Right - Left)')
    p2 = dict(data=similar, **sum_value_hue, title='P2: ΔMemorability x Overall Value\n(similar-value trials)', fname='similar_value_choices.png',
              y='chose_right', chance=0.5, ytick_step=0.05, ylabel='P(Choose Right)', dodge=0.1,
              # Li's extra title lines would shrink its axes: keep them the size of ours (4.4 x 4.2 figure, one title)
              **(dict(ylim=(None, 0.65), axes_size=(3.33, 2.79)) if USE_PILOT else {}))
    if COMPARE_PILOT:  # replication | Li: shorter and wider, legend top right of the replication panel
        p2.update(compare_figsize=(6.8, 4.3), legend_panel=0, legend_opts=dict(legend_opts, loc='upper right'))
    return [
        p2,
        dict(data=similar, **sum_value_hue, title='ΔMemorability x Overall Value, RT\n(similar-value trials)', fname='similar_value_rt.png',
             y='rt_sec', ytick_step=0.05, ylabel='RT (s)', **(dict(ylim=(None, 1.6)) if USE_PILOT else {})),
        dict(data=distant, **mem_hue, **right_value_ticks,
             # n only on gap8's (gap23's title keeps the plain subtitle)
             title=f'P1: Overall Memorability x ΔValue\n(distant-value trials'
                   + (f'; n={len(exp.included_pids)})' if exp.label == 'food_choice/gap8' else ')'),
             fname='distant_value_choices.png', y='chose_right', chance=0.5, ylabel='P(Choose Right)',
             dodge=0.1),
        dict(data=distant, **choice_mem_hue, **right_value_ticks, title='ΔValue x Overall Memorability, RT\n(distant-value trials)',
             fname='distant_value_rt.png'),
        dict(data=distant, **mem_hue, **delta_value_ticks, title='P1: Overall Memorability x ΔValue\n(distant-value trials)',
             fname='distant_value_choice_delta.png', y='chose_right', chance=0.5, ylabel='P(Choose Right)'),
        dict(data=distant, **choice_mem_hue, **delta_value_ticks, title='ΔValue x Overall Memorability, RT\n(distant-value trials)',
             fname='distant_value_rt_delta.png'),
    ]


def plot_food_figures(sources, figures_dir, distant=True):
    """sources: [(exp, label)]; the Li replication figure gets a row per source, every other figure one panel per
    source (included participants). distant=False skips the distant-value figures, which Li et al.'s design doesn't
    support."""
    os.makedirs(figures_dir, exist_ok=True)
    plot_fig3(sources, figures_dir)
    specs = [food_figures(exp) for exp, _ in sources]
    for i, spec in enumerate(specs[0]):
        if not distant and spec['fname'].startswith('distant_'):
            continue
        spec = {**FIG_DEFAULTS, **spec}
        spec.pop('data')
        plot_included([(exp, s[i]['data'], label) for (exp, label), s in zip(sources, specs)], figures_dir, **spec)


def plot_memorability_bias_both(ours, figures_dir, fname='memorability_bias_both.png', panel_size=(3.8, 4.2)):
    """Li et al.'s Figure 3 memorability-and-choice panel (low-|dvalue| trials) for our food data (left) and theirs (right)."""
    fig, axes = plt.subplots(1, 2, figsize=(2 * panel_size[0], panel_size[1] + 0.6), layout='constrained')
    for ax, exp, label in zip(axes, (ours, li_food), ('Replication (n={})', 'Li et al. (2022; n={})')):
        trials = food_trials(exp)
        data, _, kws = fig3_panels(trials)[1]
        plot_subject_lines(ax, data, x='bin', errorbar=FIG_DEFAULTS['errorbar'], **marker_style(1), **bin_ticks(5), **kws)
        ax.set_title(label.format(trials['participant_id'].nunique()), fontsize=13)
    fig.suptitle('No Overall Memorability Bias')
    if save:
        fig.savefig(os.path.join(figures_dir, fname), bbox_inches='tight')
    plt.show()


li_food = load_li_food_data()
if USE_PILOT:
    plot_food_figures([(li_food, f'Li et al. (2022; n={len(li_food.included_pids)})')], LI_FIGURES_DIR, distant=False)
else:
    for gap, exp in exp5.items():  # each version's figures in its own folder
        sources = ([(exp, f'Replication (n={len(exp.included_pids)})'),
                    (li_food, f'Li et al. (2022; n={len(li_food.included_pids)})')] if COMPARE_PILOT else [(exp, None)])
        plot_food_figures(sources, exp.figures_dir, distant=not COMPARE_PILOT)
        if COMPARE_PILOT:
            plot_memorability_bias_both(exp, exp.figures_dir)  # pilot_comparison/food_choice/<gap>/


# Other (pilot)

## THINGS stimulus memorability scores

In [ ]:
stims = pd.read_csv('../stimuli/stimuli_metadata.csv')
things = pd.read_csv('../stimuli/merged_things_metadata.csv')
stims.columns

In [ ]:
if not COMPARE_PILOT:  # comparison runs only make pilot-vs-replication figures
    # sns.histplot(things['memorability_score'])
    plt.figure(figsize=(5,4))
    plt.title('Selected image memorability scores (THINGS)')
    sns.histplot(data = stims, x = 'memorability_score', hue='memorability_bin', hue_order=['low','mid','high'], palette=['tab:blue','tab:orange','tab:green'], alpha=0.7)
    plt.xlabel('Memorability Score')
    plt.yticks([0,50,100,150])
    plt.tight_layout()
    plt.show()

## Graded value pilot

In [ ]:
if not COMPARE_PILOT:  # comparison runs only make pilot-vs-replication figures
    graded = setup_experiment('graded_value', data_root=PILOT_ROOT)

In [ ]:
def graded_figures(exp):
    graded_old = exp.df.query('old_trial == 1').dropna(subset=['optimal_old_choice']).copy()
    graded_old['rt_sec'] = graded_old['rt'] / 1000
    return [
        dict(data=graded_old, plot_fn=plot_subject_lines, title='Old Trial Choices', fname='old_choices.png',
             figsize=(5, 4), x='old_value', y='old_chosen', color=MEM_PALETTES['blue']['mid'], native_scale=True,
             xlabel='Old Card Value', ylabel='P(Choose Old Card)'),
        dict(data=graded_old, plot_fn=plot_hue_means, title='Old Trial Choices', fname='old_choices_mem.png',
             figsize=(5, 4), x='old_value', y='old_chosen', hue='memorability_bin', hue_order=MEM_ORDER,
             palette=MEM_PALETTES['blue'], **marker_style(3), dodge=True, native_scale=True, chance=0.5,
             hue_labels=MEM_LABELS, legend_title='Memorability Bin', xlabel='Old Card Value',
             ylabel='P(Choose Old Card)'),
        dict(data=graded_old, plot_fn=plot_hue_means, title='Old Trial RT', fname='old_rt_mem.png',
             x='old_chosen', y='rt_sec', hue='memorability_bin', hue_order=MEM_ORDER, palette=MEM_PALETTES['purple'],
             **marker_style(3), extra_by=['old_value'], dodge=True, hue_labels=MEM_LABELS,
             legend_title='Memorability Bin', **no_yes, ytick_step=0.05, xlabel='Chose Old Card', ylabel='RT (s)'),
    ]


if not COMPARE_PILOT:
    plot_figures(graded_figures, graded)

## Combined RT (Exps 1-3)

In [ ]:
### FIGURES: pilot | replication, old-trial RT by choice correctness x memorability, Exps 1-3 pooled

COMBINED_SHAPES = {'Exp 1': 'o', 'Exp 2': 's', 'Exp 3': '^'}  # one small marker per experiment beside each pooled mean


def combined_rt_cells(exp1, exp2, exp3):
    """Per-participant mean RT (s) in each correct x memorability cell for Exp 1, Exp 2 (value report) and Exp 3,
    centered on the participant's own mean. Included participants."""
    _, value_df = load_direct_trials(exp2)
    sources = [('Exp 1', exp1, exp1.df.query('old_trial == 1'), 'optimal_old_choice'),
               ('Exp 2', exp2, value_df, 'value_test_correct'),
               ('Exp 3', exp3, exp3.df.query('old_trial == 1'), 'optimal_old_choice')]
    frames = []
    for label, exp, df, correct_col in sources:
        d = df[df['participant_id'].isin(exp.included_pids)].copy()
        d['correct'] = pd.to_numeric(d[correct_col], errors='coerce')
        d = d.dropna(subset=['correct', 'rt']).query("memorability_bin in ['low', 'high']")
        d = d.assign(exp=label, correct=d['correct'].astype(int), rt_sec=d['rt'] / 1000,
                     pid=label + '_' + d['participant_id'].astype(str))
        frames.append(d.groupby(['exp', 'pid', 'correct', 'memorability_bin'], as_index=False)['rt_sec'].mean())
    cells = pd.concat(frames, ignore_index=True)
    cells['rt_centered'] = cells['rt_sec'] - cells.groupby('pid')['rt_sec'].transform('mean')
    return cells


def plot_combined_rt(ax, cells, dodge=0.15):
    """Big dots: mean over all participants (95% CI); small shapes: each experiment's own mean."""
    palette = {m: MEM_PALETTES['purple'][m] for m in ('low', 'high')}
    sns.pointplot(data=cells, x='correct', y='rt_centered', hue='memorability_bin', hue_order=['low', 'high'],
                  order=[0, 1], palette=palette, dodge=dodge, errorbar=FIG_DEFAULTS['errorbar'],
                  markeredgecolor='black', markeredgewidth=1, markersize=15, linewidth=3.5, ax=ax)
    for line in ax.lines:
        if line.get_marker() not in (None, '', 'None'):
            line.set_zorder(3)
    ax.get_legend().remove()
    exp_means = cells.groupby(['exp', 'correct', 'memorability_bin'])['rt_centered'].mean()
    for exp_label, shape in COMBINED_SHAPES.items():
        for mem, side in (('low', -1), ('high', 1)):
            ax.plot(np.array([0, 1]) + side * (dodge / 2 + 0.11), [exp_means[(exp_label, c, mem)] for c in (0, 1)],
                    ls='none', marker=shape, markersize=6.5, markerfacecolor=palette[mem], markeredgecolor='black',
                    markeredgewidth=0.6, zorder=2)
    ax.axhline(0, color='gray', ls='dashed', zorder=0)
    ax.yaxis.set_major_locator(MultipleLocator(0.02))
    ax.set_xticks([0, 1], labels=['Incorrect', 'Correct'])
    ax.set_xlim(-0.6, 1.6)
    ax.set_xlabel('Choice')
    return cells['pid'].nunique()


if COMPARE_PILOT:
    pilot_cells = combined_rt_cells(exp1_pilot, exp2_pilot, exp3_pilot)
    replication_cells = combined_rt_cells(exp1, exp2, exp3)
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.8), sharey=True)
    for ax, cells, label in zip(axes, [pilot_cells, replication_cells], ['Pilot', 'Replication']):
        n = plot_combined_rt(ax, cells)
        ax.set_title(f'{label} (n={n})')
    axes[0].set_ylabel('RT relative to own mean (s)')
    mem_handles = [Line2D([], [], color=MEM_PALETTES['purple'][m], marker='o', markerfacecolor=MEM_PALETTES['purple'][m],
                          markeredgecolor='black', markersize=9, lw=2.5) for m in ('low', 'high')]
    exp_handles = [Line2D([], [], ls='none', marker=shape, markersize=6.5, markerfacecolor='white', markeredgecolor='black')
                   for shape in COMBINED_SHAPES.values()]
    axes[1].legend(mem_handles + exp_handles, ['Low mem', 'High mem'] + list(COMBINED_SHAPES), loc='upper right', fontsize=10)
    fig.suptitle('Response Times - Experiments 1, 2, & 3')
    finish_fig(COMPARISON_ROOT, 'combined_rt.png')